In [ ]:
# CELDA 1: Instalar
!pip install requests beautifulsoup4 pandas lxml -q
print('Instalado!')

In [ ]:
# CELDA 2: Funciones CORREGIDAS
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import quote
import random

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0',
    'Accept': 'text/html',
    'Accept-Language': 'en-US,en;q=0.9',
}
session = requests.Session()
session.headers.update(headers)

def extraer_apellido(nombre):
    if pd.isna(nombre):
        return None
    nombre = str(nombre).strip()
    apellido = re.sub(r'^([A-Z]\.\s*)+', '', nombre).strip()
    if not apellido:
        apellido = nombre
    partes = apellido.split()
    return partes[0] if partes else apellido

def normalizar(texto):
    if not texto:
        return ''
    texto = texto.lower()
    for k, v in {'\u00e1':'a','\u00e9':'e','\u00ed':'i','\u00f3':'o','\u00fa':'u','\u00f1':'n','\u00fc':'u'}.items():
        texto = texto.replace(k, v)
    return texto

def buscar_jugador(nombre_original, equipo=None):
    try:
        apellido = extraer_apellido(nombre_original)
        if not apellido:
            return None, None
        
        apellido_norm = normalizar(apellido)
        url = 'https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query=' + quote(apellido)
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(r.content, 'lxml')
        tables = soup.find_all('table', class_='items')
        if not tables:
            return None, None
        
        rows = tables[0].find_all('tr', class_=['odd', 'even'])
        mejor_match = None
        mejor_score = 0
        
        equipo_norm = normalizar(equipo) if equipo else ''
        palabras_equipo = [p for p in equipo_norm.split() if len(p) > 3]
        
        for row in rows:
            for link in row.find_all('a'):
                href = link.get('href', '')
                if '/profil/spieler/' in href:
                    player_url = 'https://www.transfermarkt.com' + href
                    player_name = link.get_text(strip=True)
                    player_name_norm = normalizar(player_name)
                    row_text = normalizar(row.get_text())
                    
                    if apellido_norm not in player_name_norm:
                        break
                    
                    score = 0
                    for palabra in palabras_equipo:
                        if palabra in row_text:
                            score += 3
                            break
                    
                    paises = ['colombia', 'argentina', 'uruguay', 'paraguay', 'chile', 'peru', 'ecuador', 'venezuela']
                    for pais in paises:
                        if pais in row_text:
                            score += 1
                            break
                    
                    if score > mejor_score:
                        mejor_score = score
                        mejor_match = {'url': player_url, 'nombre': player_name}
                    
                    break
        
        if mejor_match and mejor_score >= 3:
            return mejor_match['url'], mejor_match['nombre']
        
        return None, None
    except Exception as e:
        return None, None

def obtener_datos(url):
    datos = {
        'transfermarkt_url': url,
        'nombre_tm': None,
        'valor_mercado': None,
        'agente': None,
        'fin_contrato': None,
        'imagen_url': None
    }
    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return datos
        
        soup = BeautifulSoup(r.content, 'lxml')
        
        # NOMBRE - limpiar numero de camiseta
        h1 = soup.find('h1', class_='data-header__headline-wrapper')
        if h1:
            nombre_raw = h1.get_text(strip=True)
            # Remover #numero del inicio
            nombre_limpio = re.sub(r'^#\d+', '', nombre_raw).strip()
            datos['nombre_tm'] = nombre_limpio
        
        # VALOR DE MERCADO - solo el valor, sin Last update
        mv = soup.find('a', class_='data-header__market-value-wrapper')
        if mv:
            # Buscar solo el texto del valor (primer hijo de texto)
            valor_text = ''
            for child in mv.children:
                if isinstance(child, str):
                    valor_text += child
                elif child.name != 'p':  # ignorar el <p> con Last update
                    valor_text += child.get_text()
            # Limpiar
            valor_text = valor_text.strip()
            # Extraer solo el valor monetario
            match = re.search(r'[\u20ac$\u00a3]?[\d.,]+[kmKM]?', valor_text)
            if match:
                datos['valor_mercado'] = match.group(0)
            else:
                datos['valor_mercado'] = valor_text.split('Last')[0].strip()
        
        # IMAGEN
        img = soup.find('img', class_='data-header__profile-image')
        if img:
            datos['imagen_url'] = img.get('src')
        
        # AGENTE - buscar correctamente
        # Buscar en la seccion de info-table
        info_table = soup.find('div', class_='info-table')
        if info_table:
            # Buscar span que diga "Agent" y luego el siguiente link
            spans = info_table.find_all('span', class_='info-table__content--regular')
            for i, span in enumerate(spans):
                if 'Agent' in span.get_text():
                    # El siguiente elemento deberia ser el agente
                    next_span = span.find_next('span', class_='info-table__content--bold')
                    if next_span:
                        agent_link = next_span.find('a')
                        if agent_link:
                            datos['agente'] = agent_link.get_text(strip=True)
                        else:
                            datos['agente'] = next_span.get_text(strip=True)
                    break
        
        # Si no encontro agente, buscar de otra forma
        if not datos['agente']:
            agent_links = soup.find_all('a', href=re.compile(r'/beraterfirma/'))
            for link in agent_links:
                text = link.get_text(strip=True)
                if text and text != 'Agent' and 'statistics' not in text.lower():
                    datos['agente'] = text
                    break
        
        # FIN DE CONTRATO - buscar en info-table
        if info_table:
            spans = info_table.find_all('span', class_='info-table__content--regular')
            for span in spans:
                text = span.get_text()
                if 'Contract expires' in text or 'contract expires' in text:
                    next_span = span.find_next('span', class_='info-table__content--bold')
                    if next_span:
                        datos['fin_contrato'] = next_span.get_text(strip=True)
                    break
        
        # Busqueda alternativa de contrato
        if not datos['fin_contrato']:
            # Buscar en el texto de la pagina
            page_text = soup.get_text()
            patterns = [
                r'Contract expires:\s*(\w+\s+\d+,\s+\d{4})',
                r'Contract expires\s*(\w+\s+\d+,\s+\d{4})',
                r'Contract expires:\s*(\d{4})',
            ]
            for pattern in patterns:
                match = re.search(pattern, page_text, re.I)
                if match:
                    datos['fin_contrato'] = match.group(1)
                    break
        
        return datos
    except:
        return datos

print('Funciones CORREGIDAS cargadas!')

In [ ]:
# CELDA 3: Cargar CSV
csv_base = 'https://docs.google.com/spreadsheets/d/e/'
csv_id = '2PACX-1vSneBjGlw2I3SyXV-uw1V8Cs_O4lbiQw39melKEZJNhunpshakPrn7AZQBN2L8N9Yw_HA-EeVOt3qvf'
csv_params = '/pub?gid=2002620668&single=true&output=csv'
CSV_URL = csv_base + csv_id + csv_params

df = pd.read_csv(CSV_URL)
print(f'Jugadores cargados: {len(df)}')
df[['Jugador', 'Equipo', 'Liga']].head(10)

In [ ]:
# CELDA 4: CONFIGURACION
DESDE = 500   # Cambiar segun la tanda
HASTA = 1000  # Cambiar segun la tanda
PAUSA_MIN = 3
PAUSA_MAX = 5

total = min(HASTA, len(df)) - DESDE
print(f'Procesara {total} jugadores (filas {DESDE} a {HASTA})')
print(f'Tiempo estimado: {total * 4 / 60:.1f} minutos')

In [ ]:
# CELDA 5: EJECUTAR SCRAPING
resultados = []
total = min(HASTA, len(df)) - DESDE

print(f'Procesando {total} jugadores...')
print('=' * 60)

for idx in range(DESDE, min(HASTA, len(df))):
    row = df.iloc[idx]
    nombre = row['Jugador']
    equipo = row.get('Equipo', '')
    liga = row.get('Liga', '')
    
    print(f'[{idx+1-DESDE}/{total}] {nombre} ({equipo})...', end=' ')
    
    url, nombre_tm = buscar_jugador(nombre, equipo)
    
    if url:
        datos = obtener_datos(url)
        datos['jugador_csv'] = nombre
        datos['equipo_csv'] = equipo
        datos['liga_csv'] = liga
        resultados.append(datos)
        print(f'OK -> {datos["nombre_tm"]}')
        print(f'         Valor: {datos["valor_mercado"]} | Contrato: {datos["fin_contrato"]} | Agente: {datos["agente"]}')
    else:
        print('NO ENCONTRADO')
        resultados.append({
            'jugador_csv': nombre,
            'equipo_csv': equipo,
            'liga_csv': liga,
            'transfermarkt_url': None,
            'nombre_tm': None,
            'valor_mercado': None,
            'agente': None,
            'fin_contrato': None,
            'imagen_url': None
        })
    
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

print('=' * 60)
encontrados = sum(1 for r in resultados if r['transfermarkt_url'])
print(f'Encontrados: {encontrados}/{total} ({encontrados/total*100:.0f}%)')

In [ ]:
# CELDA 6: Ver resultados encontrados
df_result = pd.DataFrame(resultados)
cols = ['jugador_csv', 'nombre_tm', 'equipo_csv', 'liga_csv', 'valor_mercado', 'fin_contrato', 'agente', 'transfermarkt_url', 'imagen_url']
df_result = df_result[[c for c in cols if c in df_result.columns]]

print('ENCONTRADOS:')
encontrados_df = df_result[df_result['transfermarkt_url'].notna()]
print(f'Total: {len(encontrados_df)}')
display(encontrados_df)

In [ ]:
# CELDA 7: Guardar y descargar
archivo = f'transfermarkt_{DESDE}_{HASTA}.csv'
df_result.to_csv(archivo, index=False, encoding='utf-8-sig')
print(f'Guardado: {archivo}')

from google.colab import files
files.download(archivo)

In [ ]:
# CELDA 8: TEST - Probar con un jugador conocido
print('TEST: Verificando extraccion de datos...')
test_url = 'https://www.transfermarkt.com/lionel-messi/profil/spieler/28003'
datos = obtener_datos(test_url)
for k, v in datos.items():
    print(f'{k}: {v}')